# MMF Hierarchical Reconciliation
Auto-generated by mmf-agent.

In [ ]:
%pip install /Workspace/Repos/{full_email}/many-model-forecasting[hierarchical] --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
forecast_table = "{forecast_table}"  # agnostic — confirmed in Skill 6 Step 0a
freq = "{freq}"  # H | D | W | M
date_col = "{date_col}"
target = "{target}"
reconciliation_method = "{reconciliation_method}"  # BottomUp | TopDown | MiddleOut | MinTrace | ERM

In [ ]:
import sys
sys.path.insert(0, "/Workspace/Repos/{full_email}/many-model-forecasting")
from mmf_sa import derive_hierarchy_from_unique_ids

# Auto-derive hierarchy tables if not yet present
try:
    spark.sql(f"SELECT 1 FROM {catalog}.{schema}.{use_case}_hierarchy_S LIMIT 1")
    print("✓ Hierarchy tables already exist — skipping derivation")
except Exception:
    print("Hierarchy tables not found — deriving from train_data unique_ids...")
    derive_hierarchy_from_unique_ids(
        spark=spark,
        train_table=f"{catalog}.{schema}.{use_case}_train_data",
        hierarchy_s_table=f"{catalog}.{schema}.{use_case}_hierarchy_S",
        hierarchy_tags_table=f"{catalog}.{schema}.{use_case}_hierarchy_tags",
    )
    print("✓ Hierarchy metadata derived")

In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_reconciliation

run_reconciliation(
    spark=spark,
    best_models_table=forecast_table,
    evaluation_output_table=catalog + "." + schema + "." + use_case + "_evaluation_output",
    hierarchy_s_table=catalog + "." + schema + "." + use_case + "_hierarchy_S",
    hierarchy_tags_table=catalog + "." + schema + "." + use_case + "_hierarchy_tags",
    reconciliation_output=catalog + "." + schema + "." + use_case + "_reconciliation_output",
    freq=freq,
    date_col=date_col,
    target=target,
    method=reconciliation_method,
)

In [ ]:
# Quick coherence spot-check — sample one date and verify sum of children matches parent
result = spark.table(catalog + "." + schema + "." + use_case + "_reconciliation_output")
display(result.limit(20))
print(f"Reconciliation output: {result.count()} rows")
print(f"Hierarchy levels: {[r[0] for r in result.select('hierarchy_level').distinct().collect()]}")